# Jarvis: LoRA fine-tune of Qwen2.5-0.5B-Instruct

Runs on a free Colab **T4 GPU** (Runtime > Change runtime type > T4 GPU). About 25 to 35 minutes end to end.

1. Clone the repo and generate the dataset
2. LoRA fine-tune (loss only on the assistant reply)
3. Score on `eval.jsonl` (held-out templates) and `eval_real.jsonl` (hand-written)
4. Merge, convert to GGUF, quantize to **Q4_0** (Q4_K_M is 7x slower on the tablet's no-AVX CPU)
5. Download `<OUT_NAME>-q4_0.gguf` and the report

In [ ]:
REPO = "https://github.com/mizoyyy/Surface-Go-Jarvis"
BRANCH = "main"
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
OUT_NAME = "jarvis-0.5b-v2"

!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf Surface-Go-Jarvis && git clone -q --depth 1 -b $BRANCH $REPO
%cd Surface-Go-Jarvis/training
!python generate_dataset.py && python real_eval.py
# Colab preinstalls an old torchao that new peft versions refuse to load; this notebook does not use it
!pip uninstall -y -q torchao
!pip install -q peft

In [ ]:
import json, sys, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
sys.path.insert(0, ".")
from tools import SYSTEM_PROMPT
import score

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
tok.padding_side = "left"

def load(path):
    return [json.loads(line) for line in open(path)]

def prompt_text(row):
    return tok.apply_chat_template(row["messages"][:-1], tokenize=False, add_generation_prompt=True)

def encode(row, max_len=256):
    prompt_ids = tok(prompt_text(row), add_special_tokens=False)["input_ids"]
    reply_ids = tok(row["messages"][-1]["content"] + "<|im_end|>", add_special_tokens=False)["input_ids"]
    ids = (prompt_ids + reply_ids)[:max_len]
    labels = ([-100] * len(prompt_ids) + reply_ids)[:max_len]
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": labels}

train_rows, val_rows = load("data/train.jsonl"), load("data/val.jsonl")
eval_sets = {"eval": load("data/eval.jsonl"), "eval_real": load("data/eval_real.jsonl")}
lengths = [len(encode(r)["input_ids"]) for r in train_rows]
print(len(train_rows), "train rows, max tokens", max(lengths), "mean", sum(lengths) / len(lengths))
print(prompt_text(train_rows[0]) + train_rows[0]["messages"][-1]["content"])

In [ ]:
@torch.no_grad()
def predict(model, rows, batch_size=32, max_new_tokens=80):
    model.eval()
    out = []
    for i in range(0, len(rows), batch_size):
        batch = [prompt_text(r) for r in rows[i:i + batch_size]]
        enc = tok(batch, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.pad_token_id, eos_token_id=tok.convert_tokens_to_ids("<|im_end|>"))
        out += tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return out

def evaluate(model, show_failures=8):
    results = {}
    for name, rows in eval_sets.items():
        print(f"== {name}")
        result = score.score(rows, predict(model, rows))
        score.report(result, show_failures)
        results[name] = result
    return results

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.float32).cuda()
print("Baseline (no fine-tune), expected to be poor: the tools are never described in the prompt")
baseline = evaluate(model, show_failures=3)

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
from datasets import Dataset

model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
model.print_trainable_parameters()

tok.padding_side = "right"
train_ds = Dataset.from_list([encode(r) for r in train_rows])
val_ds = Dataset.from_list([encode(r) for r in val_rows])

args = TrainingArguments(
    output_dir="lora-out", num_train_epochs=3, per_device_train_batch_size=16, gradient_accumulation_steps=1,
    learning_rate=2e-4, lr_scheduler_type="cosine", warmup_steps=50, weight_decay=0.0,
    fp16=True, logging_steps=25, eval_strategy="steps", eval_steps=100, save_strategy="no",
    report_to="none", seed=7)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100))
trainer.train()
tok.padding_side = "left"

In [ ]:
results = evaluate(model)

## Merge and export to GGUF Q4_0

If the scores above look poor, stop here and adjust the data or the training settings first.

In [ ]:
merged = model.merge_and_unload()
merged.save_pretrained("merged", safe_serialization=True)
tok.save_pretrained("merged")

!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip install -q /content/llama.cpp/gguf-py sentencepiece
!python /content/llama.cpp/convert_hf_to_gguf.py merged --outtype f16 --outfile /content/{OUT_NAME}-f16.gguf
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_NATIVE=OFF -DLLAMA_CURL=OFF -DLLAMA_BUILD_TESTS=OFF > /dev/null
!cmake --build /content/llama.cpp/build --target llama-quantize -j 4 2>&1 | tail -1
!/content/llama.cpp/build/bin/llama-quantize /content/{OUT_NAME}-f16.gguf /content/{OUT_NAME}-q4_0.gguf Q4_0 2>&1 | tail -2
!ls -la /content/*.gguf

In [ ]:
# Write a small report next to the model, for the GitHub Release notes
report = {
    "base_model": BASE_MODEL,
    "train_rows": len(train_rows),
    "baseline": {k: {"exact": v["exact"], "tool_name": v["tool_name"]} for k, v in baseline.items()},
    "finetuned": {k: {"exact": v["exact"], "tool_name": v["tool_name"], "invalid": v["invalid"],
                      "per_intent": v["per_intent"]} for k, v in results.items()},
    "train_loss": [l for l in trainer.state.log_history if "loss" in l][-1],
}
json.dump(report, open(f"/content/{OUT_NAME}-report.json", "w"), indent=2)
print(json.dumps({k: report[k] for k in ("baseline", "train_loss")}, indent=2))

from google.colab import files
files.download(f"/content/{OUT_NAME}-q4_0.gguf")
files.download(f"/content/{OUT_NAME}-report.json")